# Create fighter career statistics

Input
- `data_raw/ufc_fights.csv`

Output
- `data_clean/fighter_career_stats.csv`.

## 1. Load the fight-level data

In [1]:
import numpy as np
import pandas as pd

fights = pd.read_csv('../data_raw/ufc_fights.csv')

## 2. Create one row per fighter per fight

In [2]:
shared_cols = ['winner', 'win_by', 'last_round', 'last_round_time', 'format', 'date']

stat_cols = [
    'KD', 'SIG_STR_landed', 'SIG_STR_att',
    'TOTAL_STR_landed', 'TOTAL_STR_att',
    'TD_landed', 'TD_att', 'SUB_ATT', 'REV', 'CTRL',
    'HEAD_landed', 'HEAD_att', 'BODY_landed', 'BODY_att',
    'LEG_landed', 'LEG_att', 'DISTANCE_landed', 'DISTANCE_att',
    'CLINCH_landed', 'CLINCH_att', 'GROUND_landed', 'GROUND_att'
]

opp_stat_cols = [
    'SIG_STR_landed', 'SIG_STR_att',
    'TOTAL_STR_landed', 'TOTAL_STR_att',
    'TD_landed', 'TD_att',
    'HEAD_landed', 'HEAD_att', 'BODY_landed', 'BODY_att',
    'LEG_landed', 'LEG_att', 'DISTANCE_landed', 'DISTANCE_att',
    'CLINCH_landed', 'CLINCH_att', 'GROUND_landed', 'GROUND_att'
]

def opponent_column(column):
    return column.replace('_landed', '_opp_landed').replace('_att', '_opp_att')

def make_fighter_rows(color, opponent_color):
    columns = (
        [f'{color}_fighter', f'{opponent_color}_fighter', f'{color}_DOB']
        + shared_cols
        + [f'{color}_{column}' for column in stat_cols]
        + [f'{opponent_color}_{column}' for column in opp_stat_cols]
    )

    rows = fights[columns].copy()
    return rows.rename(columns={
        f'{color}_fighter': 'fighter_name',
        f'{opponent_color}_fighter': 'opponent',
        f'{color}_DOB': 'DOB',
        **{f'{color}_{column}': column for column in stat_cols},
        **{f'{opponent_color}_{column}': opponent_column(column) for column in opp_stat_cols}
    })

red_rows = make_fighter_rows('R', 'B')
blue_rows = make_fighter_rows('B', 'R')
fighter_fights = pd.concat([red_rows, blue_rows], ignore_index=True)

fighter_fights['winner'] = fighter_fights['winner'].fillna('').astype(str).str.strip()
fighter_fights['result'] = np.select(
    [
        fighter_fights['winner'].eq(''),
        fighter_fights['fighter_name'].eq(fighter_fights['winner'])
    ],
    ['Draw', 'Win'],
    default='Loss'
)

ordered_cols = (
    ['fighter_name', 'DOB', 'opponent', 'result', 'winner', 'win_by',
     'last_round', 'last_round_time', 'date', 'format']
    + stat_cols
    + [opponent_column(column) for column in opp_stat_cols]
)

fighter_fights = fighter_fights[ordered_cols]
fighter_fights['date'] = pd.to_datetime(fighter_fights['date'], errors='coerce')
fighter_fights = fighter_fights.sort_values(['fighter_name', 'date']).reset_index(drop=True)

## 3. Clean fight-level variables

In [3]:
win_by_map = {
    'KO/TKO': 'KO_TKO',
    "TKO - Doctor's Stoppage": 'KO_TKO',
    'Submission': 'Submission',
    'Decision - Unanimous': 'Decision',
    'Decision - Split': 'Decision',
    'Decision - Majority': 'Decision'
}
fighter_fights['win_by'] = fighter_fights['win_by'].map(win_by_map).fillna('Other')

def time_to_seconds(value):
    if pd.isna(value) or str(value).strip() in {'', '---'}:
        return 0
    minutes, seconds = str(value).strip().split(':')
    return int(minutes) * 60 + int(seconds)

def scheduled_time_seconds(value):
    if pd.isna(value) or str(value).strip() == '':
        return 0
    if value == 'No Time Limit':
        return 30 * 60
    rounds = str(value).split('(', 1)[1].split(')', 1)[0]
    return sum(int(minutes) for minutes in rounds.split('-')) * 60

fighter_fights['last_round'] = pd.to_numeric(
    fighter_fights['last_round'], errors='coerce'
).fillna(0)
fighter_fights['last_round_time'] = fighter_fights['last_round_time'].apply(time_to_seconds)
fighter_fights['total_scheduled_time'] = fighter_fights['format'].apply(scheduled_time_seconds)
fighter_fights['CTRL'] = fighter_fights['CTRL'].apply(time_to_seconds)
fighter_fights['total_fight_time'] = (
    (fighter_fights['last_round'] - 1) * 5 * 60
    + fighter_fights['last_round_time']
).clip(lower=0)
fighter_fights = fighter_fights.drop(columns='format')

ValueError: not enough values to unpack (expected 2, got 1)

## 4. Aggregate and save fighter career statistics

In [ ]:
methods = ['DEC', 'KO_TKO', 'SUB']
fighter_fights['win_method'] = fighter_fights['win_by'].map({
    'Decision': 'DEC',
    'KO_TKO': 'KO_TKO',
    'Submission': 'SUB'
}).fillna('Other')

for result in ['win', 'loss']:
    result_name = result.capitalize()
    fighter_fights[f'is_{result}'] = fighter_fights['result'].eq(result_name).astype(int)
    for method in methods:
        fighter_fights[f'{result}_{method}'] = (
            fighter_fights['result'].eq(result_name)
            & fighter_fights['win_method'].eq(method)
        ).astype(int)

striking_groups = [
    'SIG_STR', 'TOTAL_STR', 'TD', 'HEAD', 'BODY',
    'LEG', 'DISTANCE', 'CLINCH', 'GROUND'
]
fight_stat_cols = (
    ['KD', 'SUB_ATT', 'REV']
    + [f'{group}_{measure}' for group in striking_groups for measure in ['landed', 'att']]
    + [f'{group}_opp_{measure}' for group in striking_groups for measure in ['landed', 'att']]
)
numeric_cols = ['total_fight_time', 'total_scheduled_time', 'CTRL'] + fight_stat_cols
fighter_fights[numeric_cols] = (
    fighter_fights[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
)

indicator_cols = (
    ['is_win', 'is_loss']
    + [f'{result}_{method}' for result in ['win', 'loss'] for method in methods]
)
sum_cols = numeric_cols + indicator_cols
fighter_career = fighter_fights.groupby('fighter_name', as_index=False)[sum_cols].sum()
fighter_career['total_fights'] = (
    fighter_fights['fighter_name'].value_counts()
    .reindex(fighter_career['fighter_name'])
    .to_numpy()
)
fighter_career = fighter_career.rename(columns={'fighter_name': 'fighter'})
total_minutes = fighter_career['total_fight_time'] / 60

def safe_divide(numerator, denominator):
    return numerator.div(denominator).where(denominator.gt(0), 0)

fighter_career['fight_quickness'] = safe_divide(
    fighter_career['total_fight_time'], fighter_career['total_scheduled_time']
)

for result in ['win', 'loss']:
    for method in methods:
        fighter_career[f'{result}_{method}_pct'] = safe_divide(
            fighter_career[f'{result}_{method}'], fighter_career[f'is_{result}']
        )

fighter_career['CTRL_pct'] = safe_divide(
    fighter_career['CTRL'], fighter_career['total_fight_time']
)

for group in ['TOTAL_STR', 'SIG_STR', 'TD']:
    fighter_career[f'{group}_acc'] = safe_divide(
        fighter_career[f'{group}_landed'], fighter_career[f'{group}_att']
    )

for stat in ['KD', 'SUB_ATT', 'REV']:
    fighter_career[f'{stat}_per_min'] = safe_divide(
        fighter_career[stat], total_minutes
    )

for group in striking_groups:
    fighter_career[f'{group}_land_per_min'] = safe_divide(
        fighter_career[f'{group}_landed'], total_minutes
    )
    fighter_career[f'{group}_att_per_min'] = safe_divide(
        fighter_career[f'{group}_att'], total_minutes
    )
    fighter_career[f'{group}_absorbed_per_min'] = safe_divide(
        fighter_career[f'{group}_opp_landed'], total_minutes
    )
    fighter_career[f'{group}_opp_att_per_min'] = safe_divide(
        fighter_career[f'{group}_opp_att'], total_minutes
    )

for group in ['SIG_STR', 'TOTAL_STR']:
    fighter_career[f'{group}_def'] = 1 - safe_divide(
        fighter_career[f'{group}_opp_landed'], fighter_career[f'{group}_opp_att']
    )
fighter_career['TD_def'] = (
    1 - fighter_career['TD_opp_landed'].div(fighter_career['TD_opp_att'])
).where(fighter_career['TD_opp_att'].gt(0))

final_cols = [
    'fighter', 'total_fights', 'total_fight_time', 'fight_quickness',
    'win_DEC_pct', 'win_KO_TKO_pct', 'win_SUB_pct',
    'loss_DEC_pct', 'loss_KO_TKO_pct', 'loss_SUB_pct',
    'TOTAL_STR_acc', 'SIG_STR_acc', 'TD_acc', 'CTRL_pct',
    'KD_per_min', 'SUB_ATT_per_min', 'REV_per_min'
]

for group in striking_groups:
    final_cols += [
        f'{group}_land_per_min', f'{group}_att_per_min',
        f'{group}_absorbed_per_min', f'{group}_opp_att_per_min'
    ]
    if group in ['SIG_STR', 'TOTAL_STR', 'TD']:
        final_cols.append(f'{group}_def')

fighter_career_stats = fighter_career[final_cols]
fighter_career_stats.to_csv('fighter_career_stats.csv', index=False)

print(f'Saved fighter_career_stats.csv with shape {fighter_career_stats.shape}')